In [0]:
csv_path = "/Volumes/workspace/default/volume/ElectricCarData.csv"
df = spark.read.options(inferSchema=True, header=True).csv(csv_path)
df.show()
print(f"Number of rows: {df.count()}")

+-----------+--------------------+--------+------------+--------+---------------+--------------+-----------+----------+--------------+---------+-------+-----+---------+
|      Brand|               Model|AccelSec|TopSpeed_KmH|Range_Km|Efficiency_WhKm|FastCharge_KmH|RapidCharge|PowerTrain|      PlugType|BodyStyle|Segment|Seats|PriceEuro|
+-----------+--------------------+--------+------------+--------+---------------+--------------+-----------+----------+--------------+---------+-------+-----+---------+
|     Tesla |Model 3 Long Rang...|     4.6|         233|     450|            161|           940|        Yes|       AWD|    Type 2 CCS|    Sedan|      D|    5|    55480|
|Volkswagen |           ID.3 Pure|    10.0|         160|     270|            167|           250|        Yes|       RWD|    Type 2 CCS|Hatchback|      C|    5|    30000|
|  Polestar |                   2|     4.7|         210|     400|            181|           620|        Yes|       AWD|    Type 2 CCS| Liftback|      D|   

In [0]:
df.select("Brand","Model","AccelSec","TopSpeed_KmH").show(20)

+-----------+--------------------+--------+------------+
|      Brand|               Model|AccelSec|TopSpeed_KmH|
+-----------+--------------------+--------+------------+
|     Tesla |Model 3 Long Rang...|     4.6|         233|
|Volkswagen |           ID.3 Pure|    10.0|         160|
|  Polestar |                   2|     4.7|         210|
|       BMW |                iX3 |     6.8|         180|
|     Honda |                  e |     9.5|         145|
|     Lucid |                Air |     2.8|         250|
|Volkswagen |             e-Golf |     9.6|         150|
|   Peugeot |              e-208 |     8.1|         150|
|     Tesla |Model 3 Standard ...|     5.6|         225|
|      Audi |          Q4 e-tron |     6.3|         180|
|  Mercedes |      EQC 400 4MATIC|     5.1|         180|
|    Nissan |               Leaf |     7.9|         144|
|   Hyundai |Kona Electric 64 kWh|     7.9|         167|
|       BMW |                 i4 |     4.0|         200|
|   Hyundai |      IONIQ Electr

In [0]:
from pyspark.sql.functions import *

mean_AccelSec= df.select(mean("AccelSec")).collect()[0][0]
df.filter(df.AccelSec > mean_AccelSec).show()


7.396116504854368
+-----------+--------------------+--------+------------+--------+---------------+--------------+-----------+----------+--------------+---------+-------+-----+---------+
|      Brand|               Model|AccelSec|TopSpeed_KmH|Range_Km|Efficiency_WhKm|FastCharge_KmH|RapidCharge|PowerTrain|      PlugType|BodyStyle|Segment|Seats|PriceEuro|
+-----------+--------------------+--------+------------+--------+---------------+--------------+-----------+----------+--------------+---------+-------+-----+---------+
|Volkswagen |           ID.3 Pure|    10.0|         160|     270|            167|           250|        Yes|       RWD|    Type 2 CCS|Hatchback|      C|    5|    30000|
|     Honda |                  e |     9.5|         145|     170|            168|           190|        Yes|       RWD|    Type 2 CCS|Hatchback|      B|    4|    32997|
|Volkswagen |             e-Golf |     9.6|         150|     190|            168|           220|        Yes|       FWD|    Type 2 CCS|Hat

In [0]:
df.groupBy("RapidCharge").count().show()

+-----------+-----+
|RapidCharge|count|
+-----------+-----+
|        Yes|   98|
|         No|    5|
+-----------+-----+



In [0]:
from pyspark.sql import functions as F

df = df.withColumn(
    "top_speed_category",
    F.when(F.col("TopSpeed_KmH") > 150, "Fast")
     .otherwise("Slow")
)

df.show()

+-----------+--------------------+--------+------------+--------+---------------+--------------+-----------+----------+--------------+---------+-------+-----+---------+------------------+
|      Brand|               Model|AccelSec|TopSpeed_KmH|Range_Km|Efficiency_WhKm|FastCharge_KmH|RapidCharge|PowerTrain|      PlugType|BodyStyle|Segment|Seats|PriceEuro|top_speed_category|
+-----------+--------------------+--------+------------+--------+---------------+--------------+-----------+----------+--------------+---------+-------+-----+---------+------------------+
|     Tesla |Model 3 Long Rang...|     4.6|         233|     450|            161|           940|        Yes|       AWD|    Type 2 CCS|    Sedan|      D|    5|    55480|              Fast|
|Volkswagen |           ID.3 Pure|    10.0|         160|     270|            167|           250|        Yes|       RWD|    Type 2 CCS|Hatchback|      C|    5|    30000|              Fast|
|  Polestar |                   2|     4.7|         210|    

In [0]:
df = df.fillna("unknown")

In [0]:
df.write.format("delta").saveAsTable("ElectricCarData")

In [0]:
%sql
SELECT Brand
FROM ElectricCarData
WHERE top_speed_category = 'Fast';

Brand
Tesla
Volkswagen
Polestar
BMW
Lucid
Tesla
Audi
Mercedes
Hyundai
BMW
